[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RMoulla/IAO_Syn/blob/main/TP_RAG_Correction_EN.ipynb)

**Preparation:** download the English course PDF, rename it `Cours.pdf` and upload it to your Colab session. Add `OPENAI_API_KEY` to Colab Secrets and grant notebook access. Run the cells in order in a dedicated runtime.

English translation of `TP_RAG_Correction.ipynb`. The extraction, chunking, embedding, retrieval and generation code follows the original correction. The API key is loaded from Colab Secrets.


# Lab: Retrieval Augmented Generation (RAG)

Build a complete RAG pipeline: process a PDF, retrieve relevant information and use it to answer a user question.

1. **Extract and split text:** extract PDF text and divide it into fixed-size chunks.
2. **Compute embeddings and retrieve similar chunks:** identify passages relevant to a query.
3. **Generate an answer:** combine retrieved context with the user's question using `gpt-4o`.

We use `pdfplumber` for PDF extraction, `sentence-transformers` for embeddings and the OpenAI API for context-based generation.


In [ ]:
!pip install pdfplumber

## Step 1: extract and split text

Use `pdfplumber` to extract raw text and tables from the PDF. Then use Python's `textwrap.wrap()` to split the text into chunks of up to 500 characters.

These smaller chunks are used for similarity search. Extraction and chunking quality directly affect retrieval and answer quality.


In [ ]:
import pdfplumber
import textwrap

# Path to the PDF file
pdf_path = "Cours.pdf"

# Extract text and tables
full_text = ""
tables = []  # Store extracted tables

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        # Extract text
        page_text = page.extract_text()
        if page_text:
            full_text += page_text

        # Extract tables
        page_tables = page.extract_tables()
        if page_tables:
            tables.extend(page_tables)  # Append extracted tables to the list

# Split text with textwrap.wrap using a maximum character count

paragraphs = textwrap.wrap(full_text.strip(), width=500)

# Display extracted chunks
print(f"Number of extracted text chunks: {len(paragraphs)}")
for i, para in enumerate(paragraphs[:5], 1):
    print(f"Segment {i}: {para}\n")

# Display extracted tables
if tables:
    print(f"Number of extracted tables: {len(tables)}")
    print("First extracted table:")
    for row in tables[0]:
        print(row)
else:
    print("No tables found in the PDF.")

## Step 2: compute embeddings and retrieve similar chunks

1. Use `sentence-transformers` to encode each chunk as a vector representing its semantic features.
2. Compute **cosine similarity** between the query embedding and each chunk embedding.
3. Select the top `n` most similar chunks as context for the next stage.

This filters the document for passages relevant to the user's question.


In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load a pretrained sentence-transformers model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Compute an embedding for each text chunk (NumPy arrays by default)
embeddings = model.encode(paragraphs)

In [ ]:
len(embeddings[0])

In [ ]:

# User query for similarity search
query = "Which LLMs are open source?"
query_embedding = model.encode(query)

# Compute cosine similarities between the query and chunks
similarities = util.cos_sim(query_embedding, embeddings)

# Select the top-N most similar chunks
top_n = 10
top_results = similarities.topk(top_n)[1]  # Indices of the top-N similar chunks

# Retrieve top-N results by index
top_n_paragraphs = [paragraphs[i] for i in top_results[0]]

# Display the top-N most similar chunks
for i, para in enumerate(top_n_paragraphs, 1):
    print(f"Segment {i} similar: {para}\n")

In [ ]:
len(similarities)

## Step 3: generate an answer

Use the retrieved chunks and the user's question to generate a contextual answer with `gpt-4o` through the OpenAI API.

1. Join the top-ranked chunks into a single context.
2. Build system and user messages containing the instructions, context and question.
3. Call the model and display its response.

This completes RAG by combining information retrieval with answer generation. Compare the answer with the source document.


In [ ]:
!pip install openai==0.28

In [ ]:
import openai


# Read the key from Colab Secrets
from google.colab import userdata
openai.api_key = userdata.get("OPENAI_API_KEY")

# Join top-N chunks into one context
context = "\n".join(top_n_paragraphs)
prompt = f"Context:\n{context}\n\nQuestion : {query}\nAnswer:"

# Call the OpenAI API
response = openai.ChatCompletion.create(
    model="gpt-4o",  # Use the gpt-4o model
    messages=[
        {"role": "system", "content": "You specialize in retrieving information from provided documents. Answer in English using only the supplied context."},
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

# Extract and display the generated answer
generated_response = response['choices'][0]['message']['content'].strip()
print(f"Generated answer: {generated_response}")